# Task 1: Data Exploration — Understanding the Unified Schema

Ethiopia Financial Inclusion Forecasting — Selam Analytics

Confirms all three starter files load correctly and walks through the
unified schema: how `record_type` determines whether `category` or `pillar`
is populated, how `impact_link` rows connect to events via `parent_id`, and
what's actually in the enriched dataset. See `data_enrichment_log.md` for
the full account of what was added and why.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd

from src.data_loader import (
    load_unified_data, load_reference_codes, get_observations, get_events,
    get_impact_links, get_targets, events_with_impacts,
)

df = load_unified_data()
codes = load_reference_codes()
print(f"Unified dataset: {len(df)} records, {len(df.columns)} columns")
print(f"Reference codes: {len(codes)} valid (field, code) pairs")

Unified dataset: 82 records, 35 columns
Reference codes: 71 valid (field, code) pairs


## 1. Record type breakdown

In [2]:
df.record_type.value_counts()

record_type
observation    41
impact_link    25
event          13
target          3
Name: count, dtype: int64

## 2. The schema's key design principle, verified in the data

`event` rows should have `category` set and `pillar` empty (no pre-
assignment of what dimension an event affects). `observation`/`target`/
`impact_link` rows should have `pillar` set.

In [3]:
events = get_events(df)
print("Events with a non-empty pillar (should be 0):", (events.pillar.fillna("") != "").sum())
print("Events with a non-empty category (should be all):", (events.category.fillna("") != "").sum())

obs = get_observations(df)
print("Observations with an empty pillar (should be 0):", (obs.pillar.fillna("") == "").sum())

Events with a non-empty pillar (should be 0): 0
Events with a non-empty category (should be all): 13
Observations with an empty pillar (should be 0): 0


## 3. Unique indicators and their coverage

In [4]:
coverage = (
    obs.groupby(["pillar", "indicator_code", "indicator"])
    .agg(n_observations=("value_numeric", "count"),
         first_date=("observation_date", "min"),
         last_date=("observation_date", "max"))
    .reset_index()
    .sort_values(["pillar", "indicator_code"])
)
coverage

,pillar,indicator_code,indicator,n_observations,first_date,last_date
0,ACCESS,ACC_4G_COV,4G Population Coverage,2,2023-06-30,2025-06-30
1,ACCESS,ACC_ATM_DENSITY,"ATM Density (per 100,000 adults)",1,2023-12-31,2023-12-31
2,ACCESS,ACC_BRANCH_DENSITY,"Commercial Bank Branch Density (per 100,000 ad...",1,2023-12-31,2023-12-31
3,ACCESS,ACC_FAYDA,Fayda Digital ID Enrollment,4,2024-08-15,2026-01-31
4,ACCESS,ACC_MM_ACCOUNT,Mobile Money Account Rate,2,2021-12-31,2024-11-29
5,ACCESS,ACC_MM_AGENTS,Mobile Money Agent Count,1,2024-12-31,2024-12-31
6,ACCESS,ACC_MOBILE_PEN,Mobile Subscription Penetration,1,2025-12-31,2025-12-31
7,ACCESS,ACC_OWNERSHIP,Account Ownership Rate,9,2014-12-31,2024-11-29
8,AFFORDABILITY,AFF_DATA_INCOME,Data Affordability Index,1,2024-12-31,2024-12-31
9,GENDER,GEN_GAP_ACC,Account Ownership Gender Gap,3,2021-12-31,2024-11-29


## 4. The event catalog

In [5]:
events[["record_id", "category", "indicator", "observation_date", "source_type", "confidence"]].sort_values("observation_date")

,record_id,category,indicator,observation_date,source_type,confidence
33,EVT_0001,product_launch,Telebirr Launch,2021-05-17,operator,high
41,EVT_0009,policy,NFIS-II Strategy Launch,2021-09-01,regulator,high
34,EVT_0002,market_entry,Safaricom Ethiopia Commercial Launch,2022-08-01,news,high
35,EVT_0003,product_launch,M-Pesa Ethiopia Launch,2023-08-01,operator,high
36,EVT_0004,infrastructure,Fayda Digital ID Program Rollout,2024-01-01,regulator,high
37,EVT_0005,policy,Foreign Exchange Liberalization,2024-07-29,regulator,high
38,EVT_0006,milestone,P2P Transaction Count Surpasses ATM,2024-10-01,operator,high
54,EVT_0011,partnership,Safaricom-Government Fayda Enrollment Partners...,2025-06-01,news,medium
55,EVT_0012,regulation,Fayda Mandated for All Banking Transactions,2025-09-01,news,medium
39,EVT_0007,partnership,M-Pesa EthSwitch Integration,2025-10-27,operator,high


## 5. impact_link -> event join (parent_id)

This is the join pattern used throughout Tasks 2-4: every `impact_link`
points back to the event that triggered it via `parent_id`.

In [6]:
ei = events_with_impacts(df)
print(f"{len(ei)} impact_links resolve cleanly to their parent event")
ei[["event_name", "event_category", "related_indicator", "pillar", "impact_direction", "impact_magnitude"]].head(8)

25 impact_links resolve cleanly to their parent event


,event_name,event_category,related_indicator,pillar,impact_direction,impact_magnitude
0,Telebirr Launch,product_launch,ACC_MM_ACCOUNT,ACCESS,increase,high
1,Telebirr Launch,product_launch,ACC_OWNERSHIP,ACCESS,increase,medium
2,Telebirr Launch,product_launch,USG_TELEBIRR_USERS,USAGE,increase,high
3,Telebirr Launch,product_launch,USG_P2P_COUNT,USAGE,increase,high
4,Safaricom Ethiopia Commercial Launch,market_entry,ACC_MOBILE_PEN,ACCESS,increase,medium
5,Safaricom Ethiopia Commercial Launch,market_entry,ACC_4G_COV,ACCESS,increase,medium
6,M-Pesa Ethiopia Launch,product_launch,ACC_MM_ACCOUNT,ACCESS,increase,high
7,M-Pesa Ethiopia Launch,product_launch,USG_MPESA_USERS,USAGE,increase,high


## 6. Targets

In [7]:
get_targets(df)[["indicator", "indicator_code", "value_numeric", "observation_date", "source_name"]]

,indicator,indicator_code,value_numeric,observation_date,source_name
30,Account Ownership Rate,ACC_OWNERSHIP,70.0,2025-12-31,NFIS-II Strategy
31,Fayda Digital ID Enrollment,ACC_FAYDA,90000000.0,2028-12-31,Fayda/NIDP
32,Female Mobile Money Account Share,GEN_MM_SHARE,50.0,2030-12-31,NBE


## Summary

All three files (`ethiopia_fi_unified_data.csv`, `reference_codes.csv`, and
the derived helpers in `src/data_loader.py`) load and cross-reference
correctly. The schema's core principle — events stay neutral, impact_links
carry the interpretation — holds throughout the enriched 82-record dataset.
Detailed exploratory analysis continues in `02_eda.ipynb`.